# 05. Benchmark de modelos de Machine Learning

Esta etapa compara distintos algoritmos de clasificación para la concesión de crédito. La selección se realiza mediante validación cruzada estratificada sobre entrenamiento y el modelo elegido se evalúa una sola vez sobre el conjunto de prueba.

El análisis incluye rendimiento predictivo, ajuste del umbral, importancia de variables y métricas por sexo, raza y etnia.

**Entrada:** `data/processed/df_lasso2.csv`.


## Dependencias

Se usan `pandas`, `numpy`, `scikit-learn`, `xgboost`, `matplotlib`, `plotly` y utilidades de preprocesado.

Esta version refuerza el benchmark con ideas clasicas de ensemble y conjuntos desbalanceados:
- bosques con regularizacion y `oob_score`
- boosting con hiperparametros mas controlados
- pesos de clase en modelos compatibles
- metricas mas utiles que `accuracy`
- ajuste de umbral por `F1`


In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import (
    ExtraTreesClassifier,
    GradientBoostingClassifier,
    HistGradientBoostingClassifier,
    RandomForestClassifier,
)
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    precision_recall_curve,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from xgboost import XGBClassifier


In [2]:
df_final=pd.read_csv("data/processed/df_lasso2.csv")


In [3]:
TARGET_COL = "target"
RANDOM_STATE = 73
TEST_SIZE = 0.20
CV_SPLITS = 3
THRESHOLD_GRID = np.linspace(0.10, 0.90, 81)
THRESHOLD_METRIC = "f1"

# Variables sensibles exactas de tu dataset
SENSITIVE_COLS = ["sexo", "raza", "etnia"]


## Dataset de entrada

Esta celda valida la estructura del conjunto de entrada.


In [5]:
if "df_final" not in globals():
    raise NameError("No existe `df_model` en memoria. Cargalo o generalo antes de ejecutar este notebook.")

df = df_final.copy()

print(df.shape)
print(df.columns.tolist())
df.head()


(2743872, 14)
['ratio_deuda_ingresos', 'importe_prestamo', 'sistema_evaluacion_1', 'ingreso_mediano_area', 'finalidad_prestamo', 'ingresos', 'ingreso_relativo_zona', 'valor_vivienda', 'ratio_prestamo_valor', 'codigo_estado', 'sexo', 'raza', 'etnia', 'target']


,ratio_deuda_ingresos,importe_prestamo,sistema_evaluacion_1,ingreso_mediano_area,finalidad_prestamo,ingresos,ingreso_relativo_zona,valor_vivienda,ratio_prestamo_valor,codigo_estado,sexo,raza,etnia,target
0,38.0,10.915107,6,66800,31,3.713572,113.0,12.861001,69.930,PA,hombre,White,Not Hispanic or Latino,0
1,25.0,12.323860,6,97900,4,5.921578,166.0,14.224304,79.791,CA,hombre,White,Not Hispanic or Latino,1
2,33.0,13.132316,6,73100,4,6.030685,260.0,14.693061,58.638,CA,hombre,Asian,Not Hispanic or Latino,1
3,42.0,10.463132,6,64800,4,4.663439,75.0,12.524530,79.790,CA,hombre,White,Not Hispanic or Latino,1
4,55.0,11.652696,6,73100,2,4.912655,114.0,14.190517,29.235,CA,hombre,White,Not Hispanic or Latino,1


## Helpers


In [6]:
def normalize_binary_target(series):
    values = sorted(pd.Series(series).dropna().unique().tolist())
    if values not in ([0, 1], [0], [1]):
        raise ValueError(f"La variable objetivo debe ser binaria. Valores encontrados: {values}")
    return series.astype(int)


def build_feature_lists(frame, target_col):
    feature_cols = [c for c in frame.columns if c != target_col]
    numeric_cols = [c for c in feature_cols if pd.api.types.is_numeric_dtype(frame[c])]
    categorical_cols = [c for c in feature_cols if c not in numeric_cols]
    return feature_cols, numeric_cols, categorical_cols


def build_preprocessor(numeric_cols, categorical_cols):
    numeric_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    categorical_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])
    return ColumnTransformer([
        ("num", numeric_pipe, numeric_cols),
        ("cat", categorical_pipe, categorical_cols),
    ])


def build_tree_preprocessor(numeric_cols, categorical_cols):
    numeric_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
    ])
    categorical_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])
    return ColumnTransformer([
        ("num", numeric_pipe, numeric_cols),
        ("cat", categorical_pipe, categorical_cols),
    ])


def compute_class_ratio(y):
    counts = pd.Series(y).value_counts().sort_index()
    negatives = counts.get(0, 0)
    positives = counts.get(1, 0)
    pos_weight = negatives / max(positives, 1)
    minority_rate = min(negatives, positives) / max((negatives + positives), 1)
    return {
        "negatives": negatives,
        "positives": positives,
        "scale_pos_weight": pos_weight,
        "minority_rate": minority_rate,
    }


def choose_best_threshold(y_true, y_score, metric="f1", threshold_grid=None):
    if threshold_grid is None:
        _, _, thresholds = precision_recall_curve(y_true, y_score)
        threshold_grid = thresholds if len(thresholds) else np.array([0.5])

    best_threshold = 0.5
    best_value = -1
    for threshold in threshold_grid:
        y_pred = (y_score >= threshold).astype(int)
        if metric == "f1":
            value = f1_score(y_true, y_pred, zero_division=0)
        elif metric == "recall":
            value = recall_score(y_true, y_pred, zero_division=0)
        elif metric == "balanced_accuracy":
            value = balanced_accuracy_score(y_true, y_pred)
        else:
            raise ValueError(f"Metrica de umbral no soportada: {metric}")
        if value > best_value:
            best_value = value
            best_threshold = float(threshold)
    return best_threshold, best_value


def evaluate_predictions(y_true, y_score, threshold=0.5):
    y_pred = (y_score >= threshold).astype(int)
    return {
        "roc_auc": roc_auc_score(y_true, y_score),
        "average_precision": average_precision_score(y_true, y_score),
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "y_pred": y_pred,
    }


def get_model_scores(model, X):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    if hasattr(model, "decision_function"):
        raw = model.decision_function(X)
        return 1 / (1 + np.exp(-raw))
    raise ValueError("El modelo no soporta predict_proba ni decision_function.")


def cross_validate_model(name, pipeline, X_train, y_train, cv):
    scoring = {
        "roc_auc": "roc_auc",
        "average_precision": "average_precision",
        "balanced_accuracy": "balanced_accuracy",
        "precision": "precision",
        "recall": "recall",
        "f1": "f1",
    }
    cv_results = cross_validate(
        clone(pipeline),
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=1,
        return_train_score=False,
    )
    row = {"model": name}
    for metric_name in scoring:
        scores = cv_results[f"test_{metric_name}"]
        row[f"cv_mean_{metric_name}"] = scores.mean()
        row[f"cv_std_{metric_name}"] = scores.std()
    return row


def fit_final_model(name, pipeline, X_train, X_test, y_train, y_test, threshold_metric="f1"):
    model = clone(pipeline)
    model.fit(X_train, y_train)

    train_score = get_model_scores(model, X_train)
    best_threshold, train_threshold_metric = choose_best_threshold(
        y_train,
        train_score,
        metric=threshold_metric,
        threshold_grid=THRESHOLD_GRID,
    )

    test_score = get_model_scores(model, X_test)
    test_metrics = evaluate_predictions(y_test, test_score, threshold=best_threshold)
    summary = {k: v for k, v in test_metrics.items() if k != "y_pred"}
    summary["threshold"] = best_threshold
    summary[f"train_{threshold_metric}"] = train_threshold_metric

    inner_model = model.named_steps["model"]
    if hasattr(inner_model, "oob_score_"):
        summary["oob_score"] = inner_model.oob_score_

    return summary, model, test_score, test_metrics["y_pred"], best_threshold


def group_confusion_metrics(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp,
        "approval_rate_pred": (tp + fp) / max((tn + fp + fn + tp), 1),
        "tpr": tp / max((tp + fn), 1),
        "fpr": fp / max((fp + tn), 1),
        "precision": tp / max((tp + fp), 1),
    }


def compute_fairness_table(frame, sensitive_col, y_true_col="y_true", y_pred_col="y_pred"):
    rows = []
    data = frame[[sensitive_col, y_true_col, y_pred_col]].copy()
    data[sensitive_col] = data[sensitive_col].fillna("Missing").astype(str)
    for group, part in data.groupby(sensitive_col):
        metrics = group_confusion_metrics(part[y_true_col], part[y_pred_col])
        metrics["group"] = group
        metrics["count"] = len(part)
        rows.append(metrics)
    fairness_df = pd.DataFrame(rows).sort_values("count", ascending=False).reset_index(drop=True)
    if fairness_df.empty:
        return fairness_df
    ref_approval = fairness_df["approval_rate_pred"].max()
    fairness_df["disparate_impact_vs_best"] = fairness_df["approval_rate_pred"] / max(ref_approval, 1e-9)
    fairness_df["approval_rate_gap_vs_best"] = fairness_df["approval_rate_pred"] - ref_approval
    return fairness_df


## Preparacion del dataset para concesion

Se trabaja directamente con `df_model`, usando `target` como objetivo y todas las demas columnas como features.
Tambien se calcula el desbalanceo de clases para parametrizar mejor algunos modelos.


In [7]:
df[TARGET_COL] = normalize_binary_target(df[TARGET_COL])
sensitive_cols = [col for col in SENSITIVE_COLS if col in df.columns]
missing_sensitive_cols = [col for col in SENSITIVE_COLS if col not in df.columns]
feature_cols, numeric_cols, categorical_cols = build_feature_lists(df, TARGET_COL)

X = df[feature_cols].copy()
y = df[TARGET_COL].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

class_ratio = compute_class_ratio(y_train)
print(f"Sensitive columns detected: {sensitive_cols}")
if missing_sensitive_cols:
    print(f"Sensitive columns missing: {missing_sensitive_cols}")
print(f"Train: {X_train.shape} | Test: {X_test.shape}")
print(f"Class distribution train: {y_train.value_counts(normalize=True).sort_index().round(4).to_dict()}")
print(f"scale_pos_weight candidate: {class_ratio['scale_pos_weight']:.4f}")


Sensitive columns detected: ['sexo', 'raza', 'etnia']
Train: (2195097, 13) | Test: (548775, 13)
Class distribution train: {0: 0.3613, 1: 0.6387}
scale_pos_weight candidate: 0.5657


## Modelos candidatos

Se incluyen modelos importantes para clasificacion tabular y decision automatizada en riesgo de credito.
La configuración sigue tres criterios:
- bosques con regularizacion y diversidad
- boosting con control de `learning_rate`, profundidad y subsampling
- manejo del desbalanceo con pesos de clase y metricas adecuadas


In [8]:
linear_preprocessor = build_preprocessor(numeric_cols, categorical_cols)
tree_preprocessor = build_tree_preprocessor(numeric_cols, categorical_cols)

xgb_scale_pos_weight = class_ratio["scale_pos_weight"] if class_ratio["positives"] <= class_ratio["negatives"] else 1.0

model_candidates = {
    "RandomForest": Pipeline([
        ("preprocessor", tree_preprocessor),
        ("model", RandomForestClassifier(
            n_estimators=150,
            max_depth=8,
            min_samples_leaf=10,
            max_features="sqrt",
            bootstrap=True,
            oob_score=False,
            class_weight="balanced_subsample",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )),
    ]),
    "ExtraTrees": Pipeline([
        ("preprocessor", tree_preprocessor),
        ("model", ExtraTreesClassifier(
            n_estimators=150,
            max_depth=8,
            min_samples_leaf=8,
            max_features="sqrt",
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )),
    ]),
    "GradientBoosting": Pipeline([
        ("preprocessor", linear_preprocessor),
        ("model", GradientBoostingClassifier(
            n_estimators=120,
            learning_rate=0.03,
            max_depth=3,
            min_samples_leaf=20,
            subsample=0.8,
            random_state=RANDOM_STATE,
        )),
    ]),
    "HistGradientBoosting": Pipeline([
        ("preprocessor", tree_preprocessor),
        ("model", HistGradientBoostingClassifier(
            learning_rate=0.03,
            max_depth=6,
            max_iter=150,
            min_samples_leaf=20,
            l2_regularization=0.1,
            early_stopping=True,
            random_state=RANDOM_STATE,
        )),
    ]),
    "XGBoost": Pipeline([
        ("preprocessor", tree_preprocessor),
        ("model", XGBClassifier(
            n_estimators=150,
            learning_rate=0.03,
            max_depth=4,
            min_child_weight=5,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_alpha=0.2,
            reg_lambda=1.0,
            scale_pos_weight=xgb_scale_pos_weight,
            random_state=RANDOM_STATE,
            n_jobs=-1,
            eval_metric="logloss",
        )),
    ]),
}


## Metodologia de validacion

La estrategia de evaluacion se ha disenado para reducir el riesgo de sobreajuste y separar con claridad las fases de seleccion y validacion final del modelo.

En primer lugar, el conjunto de datos se divide en dos subconjuntos mediante una particion estratificada: un conjunto de entrenamiento (`train`) y un conjunto de prueba (`test`). La estratificacion permite conservar la proporcion de clases en ambos subconjuntos, algo especialmente importante cuando la variable objetivo puede presentar desequilibrio.

En segundo lugar, la comparacion entre modelos no se realiza sobre el conjunto de prueba, sino exclusivamente sobre el conjunto de entrenamiento mediante validacion cruzada estratificada de `k` particiones. Este procedimiento permite obtener una estimacion mas robusta del rendimiento medio de cada algoritmo y de su variabilidad entre particiones, evitando seleccionar un modelo por haber funcionado bien solo en una division concreta.

A continuacion, el modelo con mejor rendimiento medio en validacion se reentrena utilizando todo el conjunto de entrenamiento. Sobre este mismo conjunto se ajusta tambien el umbral de clasificacion, con el objetivo de no depender mecanicamente del valor `0.50` y adaptarlo a la metrica prioritaria del problema.

Por ultimo, el conjunto de prueba se reserva exclusivamente para la evaluacion final. De este modo, las metricas obtenidas en `test` actuan como una aproximacion mas honesta al comportamiento esperado del modelo sobre datos no vistos.

Esta metodologia permite distinguir tres niveles de analisis:
- comparacion interna de algoritmos mediante validacion cruzada
- seleccion y ajuste final del modelo en entrenamiento
- evaluacion final y analisis de sesgo sobre el conjunto de prueba

Desde el punto de vista metodologico, esta separacion es importante porque evita contaminar la fase de seleccion con informacion del conjunto final de evaluacion y refuerza la validez del analisis comparativo presentado.


## Entrenamiento y comparacion de modelos

Cada modelo se compara con validacion cruzada estratificada sobre `train`.
Solo despues se ajusta el mejor modelo sobre todo `train` y se evalua una unica vez en `test`.


In [9]:
cv = StratifiedKFold(n_splits=CV_SPLITS, shuffle=True, random_state=RANDOM_STATE)
benchmark_rows = []

print(f"Running benchmark with {len(model_candidates)} models and {CV_SPLITS} CV folds...")
for name, pipeline in model_candidates.items():
    print(f"Evaluating {name}...")
    row = cross_validate_model(name, pipeline, X_train, y_train, cv=cv)
    benchmark_rows.append(row)

benchmark_df = pd.DataFrame(benchmark_rows)
benchmark_df = benchmark_df.sort_values(
    ["cv_mean_roc_auc", "cv_mean_average_precision", "cv_mean_f1", "cv_mean_balanced_accuracy"],
    ascending=False,
).reset_index(drop=True)

cv_metric_cols = [
    "cv_mean_roc_auc",
    "cv_std_roc_auc",
    "cv_mean_average_precision",
    "cv_std_average_precision",
    "cv_mean_balanced_accuracy",
    "cv_mean_precision",
    "cv_mean_recall",
    "cv_mean_f1",
]
benchmark_df[["model", *cv_metric_cols]].style.format("{:.4f}")


Running benchmark with 5 models and 3 CV folds...
Evaluating RandomForest...
Evaluating ExtraTrees...
Evaluating GradientBoosting...
Evaluating HistGradientBoosting...


## Comparacion visual de metricas

Estas graficas reflejan rendimiento medio en validacion cruzada sobre `train`, no resultados del `test` final.


In [ ]:
fig = px.bar(
    benchmark_df,
    x="model",
    y="cv_mean_roc_auc",
    color="model",
    error_y="cv_std_roc_auc",
    text=benchmark_df["cv_mean_roc_auc"].round(4),
    title="Comparacion de modelos por ROC-AUC medio en CV",
)
fig.update_layout(showlegend=False)
fig.show()

fig_ap = px.bar(
    benchmark_df,
    x="model",
    y="cv_mean_average_precision",
    color="model",
    text=benchmark_df["cv_mean_average_precision"].round(4),
    title="Comparacion de modelos por Average Precision medio en CV",
)
fig_ap.update_layout(showlegend=False)
fig_ap.show()

metric_cols = ["cv_mean_balanced_accuracy", "cv_mean_precision", "cv_mean_recall", "cv_mean_f1"]
benchmark_long = benchmark_df.melt(id_vars="model", value_vars=metric_cols, var_name="metric", value_name="value")
fig2 = px.bar(
    benchmark_long,
    x="model",
    y="value",
    color="metric",
    barmode="group",
    title="Comparacion del resto de metricas medias en CV",
)
fig2.show()


## Modelo seleccionado y evaluacion final

Se selecciona el mejor modelo segun validacion cruzada en `train`.
Despues se entrena sobre todo `train`, se ajusta el umbral en `train` y se evalua una sola vez en `test`.


In [ ]:
selected_model_name = benchmark_df.iloc[0]["model"]
selected_pipeline = model_candidates[selected_model_name]

test_summary, selected_model, selected_scores, selected_pred, selected_threshold = fit_final_model(
    selected_model_name,
    selected_pipeline,
    X_train,
    X_test,
    y_train,
    y_test,
    threshold_metric=THRESHOLD_METRIC,
)

final_results_df = benchmark_df.copy()
for key, value in test_summary.items():
    final_results_df.loc[final_results_df["model"] == selected_model_name, f"test_{key}"] = value

print(f"Modelo seleccionado: {selected_model_name}")
print(f"Threshold seleccionado ({THRESHOLD_METRIC}): {selected_threshold:.4f}")
final_results_df


## Analisis del modelo seleccionado

Como el benchmark queda centrado en modelos de Machine Learning, el analisis principal se hace con importancias de variables.
Si algun modelo expone coeficientes, el notebook tambien los mostrara.


In [ ]:
feature_names = selected_model.named_steps["preprocessor"].get_feature_names_out()
inner_model = selected_model.named_steps["model"]

if hasattr(inner_model, "feature_importances_"):
    analysis_df = pd.DataFrame({
        "feature": feature_names,
        "importance": inner_model.feature_importances_,
    }).sort_values("importance", ascending=False)
    display(analysis_df.head(20))
    plot_col = "importance"
elif hasattr(inner_model, "coef_"):
    analysis_df = pd.DataFrame({
        "feature": feature_names,
        "coefficient": inner_model.coef_[0],
    })
    analysis_df["abs_coefficient"] = analysis_df["coefficient"].abs()
    analysis_df = analysis_df.sort_values("abs_coefficient", ascending=False)
    display(analysis_df.head(20))
    plot_col = "abs_coefficient"
else:
    analysis_df = None
    plot_col = None
    print("El modelo seleccionado no expone coeficientes o importancias directas.")


## Visualizacion de variables mas influyentes


In [ ]:
if analysis_df is not None and plot_col is not None:
    top_df = analysis_df.head(15).sort_values(plot_col, ascending=True)
    plt.figure(figsize=(10, 6))
    plt.barh(top_df["feature"], top_df[plot_col])
    plt.title(f"Top variables del modelo seleccionado: {selected_model_name}")
    plt.xlabel(plot_col)
    plt.ylabel("feature")
    plt.show()


## Matriz de confusion global


In [ ]:
cm = confusion_matrix(y_test, selected_pred)
fig_cm = px.imshow(
    cm,
    text_auto=True,
    labels={"x": "Prediccion", "y": "Real"},
    x=["No concedido (0)", "Concedido (1)"],
    y=["No concedido (0)", "Concedido (1)"],
    title=f"Matriz de confusion en test - {selected_model_name} | threshold={selected_threshold:.2f}",
)
fig_cm.show()


## Analisis de sesgo por grupos sensibles

Se calcula, para cada grupo de `sexo`, `raza` y `etnia` detectados:
- tasa de concesion predicha
- `TPR`
- `FPR`
- precision
- disparate impact frente al grupo con mayor tasa de concesion predicha

Regla practica habitual:
- `disparate impact < 0.80` puede ser una senal de alerta


In [ ]:
fairness_source = X_test.copy()
fairness_source["y_true"] = y_test.values
fairness_source["y_pred"] = selected_pred
fairness_tables = {}

for sensitive_col in sensitive_cols:
    fairness_df = compute_fairness_table(fairness_source, sensitive_col)
    fairness_tables[sensitive_col] = fairness_df
    print(f"\n### Fairness table: {sensitive_col}")
    display(fairness_df)


## Visualizacion del sesgo por grupo


In [ ]:
for sensitive_col, fairness_df in fairness_tables.items():
    if fairness_df.empty:
        continue
    fig = px.bar(
        fairness_df,
        x="group",
        y="approval_rate_pred",
        color="group",
        text=fairness_df["approval_rate_pred"].round(3),
        title=f"Tasa de concesion predicha por grupo - {sensitive_col}",
    )
    fig.update_layout(showlegend=False)
    fig.show()


## Matrices de confusion por grupo

Sirve para detectar si el modelo se equivoca mas con unos grupos que con otros.


In [ ]:
for sensitive_col in sensitive_cols:
    temp = fairness_source[[sensitive_col, "y_true", "y_pred"]].copy()
    temp[sensitive_col] = temp[sensitive_col].fillna("Missing").astype(str)
    top_groups = temp[sensitive_col].value_counts().head(4).index.tolist()

    for group in top_groups:
        part = temp[temp[sensitive_col] == group]
        cm_group = confusion_matrix(part["y_true"], part["y_pred"], labels=[0, 1])
        fig = px.imshow(
            cm_group,
            text_auto=True,
            labels={"x": "Prediccion", "y": "Real"},
            x=["No concedido (0)", "Concedido (1)"],
            y=["No concedido (0)", "Concedido (1)"],
            title=f"Matriz de confusion - {sensitive_col} = {group}",
        )
        fig.show()


## Resumen final

Cosas que debes mirar al cerrar el analisis:
- si el mejor modelo en `cv_mean_roc_auc` tambien aguanta bien en `cv_mean_average_precision`, `cv_mean_balanced_accuracy`, `cv_mean_recall` y `cv_mean_f1`
- si la desviacion en validacion cruzada es alta, porque eso indica inestabilidad entre folds
- si el rendimiento final en `test` cae mucho respecto a la validacion cruzada, porque eso sugiere sobreajuste
- si el umbral optimizado cambia mucho respecto a `0.50`, porque eso suele indicar que el corte por defecto no era adecuado
- si algun grupo de `sexo`, `raza` o `etnia` recibe una tasa de concesion mucho menor
- si algun grupo tiene `FPR` o `TPR` claramente peor que otros
- si aparece `disparate impact` por debajo de `0.80`
- si las variables sensibles o sus proxies aparecen entre las variables mas influyentes

Metodologia de evaluacion en esta version:
- comparacion de modelos con validacion cruzada estratificada en `train`
- seleccion del mejor modelo usando solo metricas de validacion
- entrenamiento final sobre todo `train`
- evaluacion final una sola vez sobre `test`

Si detectas diferencias fuertes entre grupos, eso no prueba por si solo discriminacion legal, pero si justifica una revision mas profunda del proceso, del umbral de decision y de las variables usadas.
